# Day 13 — Production engineering: traces, retries, idempotency, cost & latency
**Theory:** [day-13-production-engineering.md](../theory/day-13-production-engineering.md) · ⏱ ~75 min · Level: medium

**You will:** trace every step of the capstone agent, build a cost/latency report, add retries with backoff,
prove refunds are idempotent, and apply cost levers one at a time.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Trace a conversation

In [ ]:
import time, random, statistics, sqlite3
import pandas as pd
from shopease.agent import SupportAgent
from shopease.tracing import Tracer

setup_data(force=True, quiet=True)
tracer = Tracer()
agent = SupportAgent("C002", tracer=tracer)
reply = agent.chat("My running shoes O2003 never arrived. What can I do?")
print(reply.text, "\n")
print(tracer.pretty())

## 2. A small cost & latency report over mixed traffic

In [ ]:
TRAFFIC = [("C001", "Where is my smart watch?"), ("C005", "How long does express shipping take?"),
           ("C003", "Can I return O2004?"), ("C006", "I want to speak to a manager"),
           ("C002", "Refund my earbuds O2006"), ("C004", "What's your warranty on electronics?")]
def report(level=None):
    rows = []
    for cid, msg in TRAFFIC:
        t0 = time.time()
        r = SupportAgent(cid, thinking_level=level).chat(msg)
        rows.append({"route": r.route, "steps": r.steps, "seconds": round(time.time() - t0, 2), "≈cost $": round(r.cost, 5), "msg": msg})
    df = pd.DataFrame(rows)
    secs = sorted(df.seconds)
    print(f"thinking={level or 'model default'}: total ${df['≈cost $'].sum():.4f} · p50 {statistics.median(secs):.1f}s · "
          f"p95 {secs[min(len(secs) - 1, int(0.95 * len(secs)))]:.1f}s")
    return df
baseline = report(); baseline

## 3. Lever: thinking level per route
Lower thinking on the agent path. Only keep it if quality holds (check the answers, then the eval suite).

In [ ]:
low = report("low"); low[["msg", "≈cost $", "seconds"]].assign(baseline_cost=baseline["≈cost $"])

## 4. Retries with exponential backoff + jitter (no API needed)

In [ ]:
calls = {"n": 0}
def flaky_shipping_api(order_id):
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("upstream timeout")
    return f"{order_id}: in transit, last scan Pune hub"

def with_retries(fn, *args, attempts=4, base=0.2):
    for i in range(attempts):
        try:
            return fn(*args)
        except (TimeoutError, ConnectionError) as e:
            if i == attempts - 1:
                raise
            delay = base * 2 ** i + random.uniform(0, base)
            print(f"attempt {i + 1} failed ({e}); retrying in {delay:.2f}s")
            time.sleep(delay)

print(with_retries(flaky_shipping_api, "O2003"))

## 5. Idempotency: retries must not refund twice

In [ ]:
from shopease.tools import Session, issue_refund
s = Session("C004", conversation_id="conv-42")
print(issue_refund(s, "O2007", "not needed"))
print(issue_refund(s, "O2007", "not needed"))   # a retry or a looping agent
print(sqlite3.connect("projects/shopease-support-agent/data/shopease.db").execute(
      "SELECT order_id, status, idempotency_key FROM refunds").fetchall())

## 6. Cascade math: try cheap first, escalate if needed

In [ ]:
def cascade_cost(c_small, c_big, handled):
    return c_small + (1 - handled) * c_big
for a in (0.5, 0.7, 0.9):
    e = cascade_cost(0.002, 0.04, a)
    print(f"cheap path handles {a:.0%}: ${e:.4f} vs ${0.04:.4f} → saving {1 - e / 0.04:.0%}")

## 7. Batch API: ~50% off for offline work (evals, backfills)
Results arrive asynchronously (minutes, up to 24h). Submit here, check back later.

In [ ]:
tickets = ["Where is my order?", "Do you ship to India?", "My watch broke", "Refund please", "Thanks, all good!"]
requests = [{"contents": [{"role": "user", "parts": [{"text": f"Classify as faq/order_issue/other, one word: {t}"}]}],
             "metadata": {"key": f"t{i}"}} for i, t in enumerate(tickets)]
try:
    job = client.batches.create(model=CHEAP_MODEL, src=requests, config={"display_name": "ticket-classifier"})
    print(job.name, job.state)
except Exception as e:
    job = None
    print(f"Batch API not available on your tier/model ({type(e).__name__}: {str(e)[:150]})")

In [ ]:
# Run this cell again later. Inline results come back in request order; key them by your metadata.
if job:
    job = client.batches.get(name=job.name)
    print(job.state)
    if job.state.name == "JOB_STATE_SUCCEEDED" and job.dest and job.dest.inlined_responses:
        for t, res in zip(tickets, job.dest.inlined_responses):
            print(f"{t:<25} → {text_of(res.response) if res.response else res.error}")

## ✏️ Exercises
1. Send the trace spans to Langfuse (free tier) with OpenTelemetry. Which attributes would you filter dashboards by?
2. Add a run-level deadline to `SupportAgent` (e.g. 30 s) and return a graceful message when it's hit.
3. Re-run the eval suite after switching to `thinking_level="low"`. Did quality hold? Decide and write down why.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None